# Phase 2: Probability and Probability Distributions

## 1. Introduction

The objective of this phase is to translate the project's research question into probabilistic terms and examine the probability distributions of selected health-related variables.

This analysis uses data from the NHANES 2021–2023 survey, focusing on adults aged 20 years and older. The unit of observation is an individual participant.

The main focus is the relationship between depressive symptoms, measured using the PHQ-9 questionnaire, and obesity, measured using Body Mass Index (BMI). Additional quantitative health measurements, including waist circumference and blood pressure, are also available for the analysis.

This phase covers empirical probabilities, conditional probabilities, independence, discrete and continuous random variables, binomial and normal distributions, expected value, variance, and covariance.

Because NHANES uses a complex survey sampling design, unweighted sample proportions and weighted population estimates will be distinguished throughout the analysis.

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyreadstat
from scipy import stats

demo, demo_meta = pyreadstat.read_xport(
    "../data/raw/DEMO_L.xpt",
    encoding="WINDOWS-1252"
)

dpq, dpq_meta = pyreadstat.read_xport(
    "../data/raw/DPQ_L.xpt",
    encoding="WINDOWS-1252"
)

bmx, bmx_meta = pyreadstat.read_xport(
    "../data/raw/BMX_L.xpt",
    encoding="WINDOWS-1252"
)

bpxo, bpxo_meta = pyreadstat.read_xport(
    "../data/raw/BPXO_L.xpt",
    encoding="WINDOWS-1252"
)

In [6]:
# Restrict DEMO to adults aged 20 and above
demo_adults = demo[demo["RIDAGEYR"] >= 20].copy()

# Merge datasets using participant ID (SEQN)
analysis_data = (
    demo_adults
    .merge(dpq, on="SEQN", how="inner", validate="one_to_one")
    .merge(bmx, on="SEQN", how="inner", validate="one_to_one")
    .merge(bpxo, on="SEQN", how="inner", validate="one_to_one")
)

print("Merged dataset shape:", analysis_data.shape)
print("Unique participants:", analysis_data["SEQN"].nunique())

Merged dataset shape: (6064, 69)
Unique participants: 6064


In [7]:
# Variables needed for Phase 2
phq_items = [
    "DPQ010", "DPQ020", "DPQ030",
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]

systolic_items = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
diastolic_items = ["BPXODI1", "BPXODI2", "BPXODI3"]

selected_columns = [
    "SEQN",
    "RIDAGEYR",
    "RIAGENDR",
    "WTMEC2YR",
    "SDMVSTRA",
    "SDMVPSU",
    "BMXBMI",
    "BMXWAIST",
    *phq_items,
    *systolic_items,
    *diastolic_items
]

phase2_data = analysis_data[selected_columns].copy()

# Calculate PHQ-9 only when all nine answers are valid
valid_answers = phase2_data[phq_items].where(
    phase2_data[phq_items].isin([0, 1, 2, 3])
)

phase2_data["PHQ9"] = valid_answers.sum(axis=1, min_count=9)

# Calculate average blood pressure from available readings
phase2_data["Mean_Systolic"] = phase2_data[systolic_items].mean(axis=1)
phase2_data["Mean_Diastolic"] = phase2_data[diastolic_items].mean(axis=1)

print("Phase 2 dataset shape:", phase2_data.shape)

Phase 2 dataset shape: (6064, 26)


## 2. Definition of Events and Random Variables

The following events and random variables are defined at the individual participant level.

### 2.1. Events

Let the sample space consist of eligible adult participants in the merged NHANES dataset.

- **Event A — Obesity:** A participant is classified as obese if their BMI is greater than or equal to 30 kg/m².

- **Event B — Moderate-or-higher depressive symptom severity:** A participant belongs to this event if their total PHQ-9 score is greater than or equal to 10.

The PHQ-9 score is calculated only when all nine questionnaire items have valid responses. A score of 10 or higher is used as a screening threshold for moderate-or-higher depressive symptom severity; it does not constitute a clinical diagnosis.

Both events are binary categorical events, with each participant either belonging or not belonging to each event.

### 2.2. Random Variables

- **X — BMI:** The BMI value of a randomly selected eligible participant. This is a quantitative continuous random variable, measured in kg/m².

- **Y — PHQ-9 score:** The total PHQ-9 score of a randomly selected participant with all nine valid questionnaire responses. This is a discrete random variable taking integer values from 0 to 27.

### 2.3. Analysis Population

For calculations involving events A and B jointly, only participants with valid BMI and PHQ-9 values are included. This ensures that marginal, joint, and conditional probabilities are calculated using a consistent denominator.

Missing values are not treated as event occurrences or non-occurrences; participants with missing values in the variables required for a particular calculation are excluded from that calculation.

In [8]:
# Keep participants with valid BMI and PHQ-9
prob_data = phase2_data.dropna(
    subset=["BMXBMI", "PHQ9"]
).copy()

# Define events A and B
prob_data["A_Obesity"] = prob_data["BMXBMI"] >= 30
prob_data["B_PHQ9"] = prob_data["PHQ9"] >= 10

# Marginal probabilities
P_A = prob_data["A_Obesity"].mean()
P_B = prob_data["B_PHQ9"].mean()

# Joint probability
P_AB = (
    prob_data["A_Obesity"] &
    prob_data["B_PHQ9"]
).mean()

# Conditional probabilities
P_B_given_A = P_AB / P_A
P_A_given_B = P_AB / P_B

print("Number of participants:", len(prob_data))
print("P(A) - Obesity:", P_A)
print("P(B) - PHQ-9 >= 10:", P_B)
print("P(A and B):", P_AB)
print("P(B | A):", P_B_given_A)
print("P(A | B):", P_A_given_B)

# Independence check
print("\nIndependence check:")
print("P(A) * P(B):", P_A * P_B)
print("P(A and B):", P_AB)

Number of participants: 5164
P(A) - Obesity: 0.4138264910921766
P(B) - PHQ-9 >= 10: 0.12509682416731216
P(A and B): 0.06022463206816421
P(B | A): 0.14553111839026672
P(A | B): 0.48142414860681115

Independence check:
P(A) * P(B): 0.05176837979193379
P(A and B): 0.06022463206816421


### 2.4. Empirical Marginal, Joint, and Conditional Probabilities

To estimate the probabilities of events A and B, participants with missing BMI or PHQ-9 values are excluded from this calculation.

The empirical marginal probabilities are estimated using the observed relative frequencies of each event:

- P(A): The proportion of participants classified as obese.
- P(B): The proportion of participants with PHQ-9 scores of 10 or higher.

The joint probability P(A ∩ B) is calculated as the proportion of participants who satisfy both conditions simultaneously.

Two conditional probabilities are then calculated:

- P(B | A): The probability of moderate-or-higher depressive symptom severity among participants classified as obese.
- P(A | B): The probability of obesity among participants with moderate-or-higher depressive symptom severity.

Finally, independence is examined by comparing P(A ∩ B) with P(A) × P(B). If these quantities are approximately equal, the observed probabilities are compatible with independence. Otherwise, the difference suggests an association between the two events.

This is a descriptive probabilistic assessment, not a formal statistical test of independence.

In [9]:
# Calculate P(A | B) using Bayes' theorem
P_A_given_B_bayes = (P_B_given_A * P_A) / P_B

print("Direct calculation of P(A | B):", P_A_given_B)
print("Bayes' theorem calculation:", P_A_given_B_bayes)

print(
    "Absolute difference:",
    abs(P_A_given_B - P_A_given_B_bayes)
)

Direct calculation of P(A | B): 0.48142414860681115
Bayes' theorem calculation: 0.48142414860681115
Absolute difference: 0.0


### Bayes' Theorem

Let:

- $A$: A participant is classified as obese (BMI $\geq 30$).
- $B$: A participant has moderate-to-severe depressive symptoms (PHQ-9 $\geq 10$).

Bayes' theorem expresses the conditional probability of obesity given depressive symptoms as:

$$
P(A\mid B)=\frac{P(B\mid A)\times P(A)}{P(B)}
$$

In our analysis, these probabilities have already been estimated from the empirical frequencies. Therefore, Bayes' theorem provides an alternative way to express the conditional probability calculated earlier using the joint probability.

The two approaches should produce the same result, apart from possible minor differences due to numerical rounding. This illustrates how conditional probabilities can be reversed using the joint probability and the marginal probabilities.

All probabilities here are unweighted empirical estimates from the analyzed sample.

In [10]:
# Keep participants with valid BMI and gender
gender_data = phase2_data.dropna(
    subset=["BMXBMI", "RIAGENDR"]
).copy()

# Define obesity event
gender_data["Obesity"] = gender_data["BMXBMI"] >= 30

# Calculate conditional probabilities by gender
gender_probabilities = (
    gender_data
    .groupby("RIAGENDR")["Obesity"]
    .agg(["count", "sum", "mean"])
)

# Add readable group labels
gender_probabilities.index = gender_probabilities.index.map({
    1: "Male",
    2: "Female"
})

gender_probabilities = gender_probabilities.rename(
    columns={
        "count": "Number of participants",
        "sum": "Number of obese participants",
        "mean": "Empirical probability of obesity"
    }
)

print(gender_probabilities)

          Number of participants  Number of obese participants  \
RIAGENDR                                                         
Male                        2680                          1026   
Female                      3290                          1429   

          Empirical probability of obesity  
RIAGENDR                                    
Male                              0.382836  
Female                            0.434347  


### Conditional Probability of Obesity by Sex

To examine whether the empirical probability of obesity differs between males and females, participants were divided into two groups according to the NHANES variable `RIAGENDR`. Obesity was defined as having a BMI greater than or equal to 30:

$$
A = \{\text{BMI} \geq 30\}
$$

The conditional probability of obesity was calculated separately for each group:

$$
P(A \mid \text{Male}) =
\frac{\text{Number of obese male participants}}
{\text{Total number of male participants}}
$$

$$
P(A \mid \text{Female}) =
\frac{\text{Number of obese female participants}}
{\text{Total number of female participants}}
$$

The results are presented below:

| Sex | Number of participants | Number of obese participants | Empirical probability of obesity |
|---|---:|---:|---:|
| Male | 2,680 | 1,026 | 0.3828 (38.28%) |
| Female | 3,290 | 1,429 | 0.4343 (43.43%) |

The empirical probability of obesity was approximately **38.28% among males** and **43.43% among females**. Thus, within this sample, the observed probability of obesity was about **5.15 percentage points higher among females**.

This illustrates how conditioning on a subgroup changes the probability being examined. Instead of considering the overall probability of obesity among all participants, we calculate the relative frequency of obesity within each sex group.

These results describe an association in the analyzed sample. They do not establish that sex causes differences in obesity, nor do they indicate whether the observed difference is statistically significant. Furthermore, these are unweighted empirical probabilities; therefore, they should not automatically be interpreted as population-level estimates for all adults in the United States.